!nvidia-smi --query-gpu=name,power.draw --format=csv


In [ ]:
!pkill -f "ollama serve"; pkill -f cloudflared; echo cleaned

In [ ]:
!nvidia-smi --query-gpu=name,power.draw --format=csv


In [ ]:
!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, os, time
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_KEEP_ALIVE="30m", OLLAMA_MAX_LOADED_MODELS="4")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

In [ ]:
for m in ["qwen2.5:0.5b", "llama3.2:1b", "llama3.2:3b", "llama3.1:8b"]:
    !ollama pull {m}
!ollama list

In [ ]:
import threading, json
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

SAMPLES = []  # (ts, watts, util)

def sampler():
    while True:
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=power.draw,utilization.gpu",
                                  "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
            w, u = [float(x) for x in out.strip().split(",")]
            SAMPLES.append((time.time(), w, u))
            del SAMPLES[:-2400]  # keep about 20 min
        except Exception:
            pass
        time.sleep(0.5)

class H(BaseHTTPRequestHandler):
    def do_GET(self):
        u = urlparse(self.path)
        if u.path == "/power" and SAMPLES:
            ts, w, util = SAMPLES[-1]
            body = {"ts": ts, "now": time.time(), "watts": w, "util": util, "device": "colab-t4"}
        elif u.path == "/power/history":
            since = float(parse_qs(u.query).get("since", ["0"])[0])
            body = {"now": time.time(), "device": "colab-t4", "samples": [s for s in SAMPLES if s[0] >= since]}
        else:
            self.send_response(404); self.end_headers(); return
        data = json.dumps(body).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json"); self.end_headers()
        self.wfile.write(data)
    def log_message(self, *a): pass

threading.Thread(target=sampler, daemon=True).start()
threading.Thread(target=ThreadingHTTPServer(("0.0.0.0", 8005), H).serve_forever, daemon=True).start()
time.sleep(2); print(SAMPLES[-1])

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re

def tunnel(port, host_header=None):
    args = ["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{port}"]
    if host_header: args += ["--http-host-header", host_header]
    subprocess.Popen(args, stdout=open(f"cf_{port}.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"cf_{port}.log").read())
        if m: return m.group(0)
    raise RuntimeError(f"no tunnel for {port}, check cf_{port}.log")

OLLAMA_URL = tunnel(11434, "localhost:11434")
POWER_URL = tunnel(8005)
print(f"OLLAMA_URL={OLLAMA_URL}\nPOWER_URL={POWER_URL}")

In [ ]:
import requests, statistics
prompts = ["What's 15% of 80?", "Capital of Peru?", "Define photosynthesis in one line.", "Translate 'thank you' to French.",
           "Explain how a heat pump works.", "Write a haiku-free two-line poem about rain.", "Summarize the plot of Hamlet briefly.",
           "What is 17 * 23?", "Give 3 tips for a job interview.", "Why is the sky blue?"] * 2

def joules(model, prompt):
    t0 = time.time()
    requests.post("http://localhost:11434/api/generate", json={"model": model, "prompt": prompt, "stream": False})
    t1 = time.time()
    s = [w for ts, w, _ in SAMPLES if t0 <= ts <= t1] or [SAMPLES[-1][1]]
    return statistics.mean(s) * (t1 - t0)

for m in ["qwen2.5:0.5b", "llama3.1:8b"]:
    joules(m, "hi")  # warm up
small = statistics.mean(joules("qwen2.5:0.5b", p) for p in prompts)
big = statistics.mean(joules("llama3.1:8b", p) for p in prompts)
print(f"Gear 1: {small:.0f} J/answer | Gear 4: {big:.0f} J/answer | ratio {big/small:.1f}x")